In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
print("Imported!")

In [ ]:
book_scraper_url = "https://books.toscrape.com/catalogue/category/books"
genre_to_scrape = [
    "travel_2",
    "mystery_3",
    "historical-fiction_4",
    "sequential-art_5",
    "classics_6"
]

def scrape_books_for_genre(html: str, genre: str):
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception as e:
        print(f"Error parsing HTML for genre {genre}: {e}")
        return []
    all_books = soup.find_all("article", class_="product_pod")
    book_jsons = []
    for book in all_books:
        book_json = {
            "title": book.h3.a["title"],
            "category": genre,
            "price_in_gbp": book.find("p", class_="price_color").text.strip(),
            "star_rating": book.find("p", class_="star-rating").get("class", [])[-1],
            "availability": book.find("p", class_="availability").text.strip()
        }
        book_jsons.append(book_json)
    return book_jsons

def scrape_all_books():
    books = []
    for genre in genre_to_scrape:
        genre_url = f"{book_scraper_url}/{genre}/index.html"
        print(f"Scraping genre: {genre}")
        response = requests.request("GET", genre_url)
        if response.status_code == 200:
            books.extend(scrape_books_for_genre(response.text, genre))
    print(f"Total books scraped: {len(books)}")
    return books

print("###################################### TASK 1 ######################################")
books = scrape_all_books()
df = pd.DataFrame(books)
print(df.head(10))

In [ ]:
def clean_data(df):
  star_rating = {
    "One":1,
    "Two":2,
    "Three":3,
    "Four":4,
    "Five":5
  }
  #removing numbers at the end of category column
  df['category'] = df['category'].str.replace(r'_\d+','', regex=True)
  #Converting its dtype,word-to-number and filling null values with median
  df["star_rating"] = df["star_rating"].map(star_rating)
  df['star_rating'] = df['star_rating'].fillna(df['star_rating'].median())
  # Removing symbols in price column and filling null values with mean
  df["price_in_gbp"] = df["price_in_gbp"].str.replace("Â£", "").astype(float, errors='raise')
  df["price_in_gbp"] = df["price_in_gbp"].fillna(df["price_in_gbp"].mean())
  # converting availability column to bool
  df['availability'] = df['availability'].str.contains("In stock", case="False", na=False)
  return df
print("###################################### TASK 2 ######################################")
df = clean_data(df)
print(df.head())

In [ ]:
def gbp_to_inr(books):
  to_inr = 105.50
  df['price_in_inr'] = df['price_in_gbp'] * to_inr
  return df
print("###################################### TASK 3 ######################################")
df = gbp_to_inr(books)
print(df.head(10))

In [ ]:
def database():
  connection = sqlite3.connect("books.db")
  cursor = connection.cursor()
  cursor.execute("DROP TABLE IF EXISTS books")
  cursor.execute("DROP TABLE IF EXISTS categories")
  print("Creating categories table")
  cursor.execute("""
             CREATE TABLE IF NOT EXISTS categories(
             category_id INTEGER PRIMARY KEY AUTOINCREMENT,
             category_name TEXT NOT NULL UNIQUE)
             """)
  print('Table 1 created')
  print("Creating books table")
  cursor.execute("""
                CREATE TABLE IF NOT EXISTS books(
                book_id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                price_in_gbp REAL NOT NULL,
                price_in_inr REAL NOT NULL,
                rating INTEGER NOT NULL,
                availability BOOL NOT NULL,
                category_id INTEGER NOT NULL,
                FOREIGN KEY(category_id) REFERENCES categories(category_id))
                """)
  print('Table 2 created')
  print("Database Created Successfully")
  return cursor
print("###################################### TASK 4 ######################################")
cursor = database()

In [ ]:
def category(cursor, books):
  inserted_categories=[]
  unique_categories = books['category'].unique()
  for i, category in enumerate(unique_categories):
    inserted_categories.append((i, category))
  cursor.execute("DELETE FROM categories")
  cursor.executemany("INSERT INTO categories(category_id,category_name) VALUES (?,?)", inserted_categories)
  print(f"{len(inserted_categories)} categories inserted")
category(cursor, df)

In [ ]:
def insert_books(cursor, books):
  categories = {
      "travel": 0,
      "mystery": 1,
      "historical-fiction": 2,
      "sequential-art": 3,
      "classics": 4
  }
  cursor.execute("DELETE FROM books")
  query="""
    INSERT INTO books(
      title,
      price_in_gbp,
      price_in_inr,
      rating,
      availability,
      category_id
    ) VALUES (?,?,?,?,?,?)"""
  book_data=[
      (
          book["title"],
          book["price_in_gbp"],
          book["price_in_inr"],
          book["star_rating"],
          book["availability"],
          categories[book["category"]]
      )
      for i, book in books.iterrows()
  ]
  cursor.executemany(query, book_data)
  print(f"{len(book_data)} books inserted")
insert_books(cursor, df)

In [ ]:
# Applying queries
query1 = """SELECT title FROM books WHERE rating = 5"""
result1 = cursor.execute(query1).fetchall()
print(result1)
print(f"There are {len(result1)} books with 5-star rating")
query2 = """SELECT title,price_in_gbp FROM books ORDER BY price_in_gbp DESC LIMIT 1"""
result2=cursor.execute(query2).fetchall()
print(f"The most expensive book is :")
print(result2)
query3="""
SELECT DISTINCT
categories.category_name,title
FROM books
JOIN categories
ON books.category_id=
categories.category_id
ORDER BY books.price_in_gbp DESC
LIMIT 5 """
result3=cursor.execute(query3).fetchall()
print("Categories and names of top-5 expensive books")
print(result3)
query4="""SELECT title,price_in_gbp FROM books WHERE price_in_gbp BETWEEN 10 AND 20"""
result4=cursor.execute(query4).fetchall()
print(f"There are {len(result4)} books with prices between 10 and 20 GBP")
print(result4)
query5="""SELECT books.title,
          books.price_in_gbp
         FROM books
      JOIN categories
      ON books.category_id =
      categories.category_id
      WHERE categories.category_name =
      'mystery'
      """
result5 = cursor.execute(query5).fetchall()
print(f"There are {(len(result5))} books from mystery category")
print(result5)
 

In [ ]:
df_books = pd.read_sql("SELECT * FROM books", cursor.connection)
df_categories = pd.read_sql("SELECT * FROM categories", cursor.connection)
query_join = """
  SELECT books.title, categories.category_name,price_in_gbp
  FROM books
  JOIN categories ON books.category_id = categories.category_id
"""
df_sql_join = pd.read_sql(query_join, cursor.connection)
df_pandas_merge = pd.merge(df_books, df_categories, on = "category_id")
print("sql join result")
print(df_sql_join.head())
print("pandas merge result")
print(df_pandas_merge.head())